In [1]:
import numpy as np
from scipy.special import comb
from scipy.linalg import expm

from electron_integrals import *

from CI import *

from quantum_systems import GeneralOrbitalSystem, ODQD
from configuration_interaction import CISD

Define number of electrons and orbitals

In [45]:
# Number of orbitals (without spin)
num_orbitals = 6
# Number of electrons
num_electrons = 2
#Include spin?
include_spin = True

num_spin_orbitals = (1+int(include_spin))*num_orbitals

Calculate electron integrals

In [46]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

#pot = GaussianWell(w=100, a=1, center=0)
pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a=0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

#Chemistry convention... (Following Hochstuhl)
g = g.transpose(0,2,1,3)

#g = g - g.transpose(0, 1, 3, 2) anti-symmetrisation, code further down not written for this

print('Sanity test, due to symmetry in g this should be zero:')
print(-g[2,1,1,0]+g[2,1,0,1]+g[1,2,1,0]-g[1,2,0,1])

Sanity test, due to symmetry in g this should be zero:
0.0


Solve using address scheme

In [47]:
H = AddressHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E, Psi = np.linalg.eigh(H)
print(E)

[ 2.69518071  2.73313865  2.73313865  2.73313865  3.70897992  3.73641856
  3.73641856  3.73641856  4.57540917  4.64307892  4.64307892  4.64307892
  4.72949421  4.74023615  4.74023615  4.74023615  5.62018958  5.6594825
  5.6594825   5.6594825   5.73763097  5.77939444  5.77939444  5.77939444
  6.51430347  6.5947808   6.5947808   6.5947808   6.69232757  6.70261452
  6.70261452  6.70261452  6.77898812  6.77898812  6.77898812  7.62404927
  7.64612613  7.64612613  7.64612613  7.76173783  7.76173783  7.76173783
  8.51540735  8.57938471  8.57938471  8.57938471  8.73608255  8.73608255
  8.73608255  9.42295247  9.69793715  9.69793715  9.69793715 10.01790688
 10.56473012 10.62821763 10.62821763 10.62821763 10.70969514 11.02184359
 11.27609891 11.62936661 12.52912408 13.41298761 14.56514541 15.42614026]


Solve using Slater-Condon 

In [48]:
H_sc = SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_sc, _ = np.linalg.eigh(H_sc)
print(E_sc)

[ 2.69518071  2.73313865  2.73313865  2.73313865  3.70897992  3.73641856
  3.73641856  3.73641856  4.57540917  4.64307892  4.64307892  4.64307892
  4.72949421  4.74023615  4.74023615  4.74023615  5.62018958  5.6594825
  5.6594825   5.6594825   5.73763097  5.77939444  5.77939444  5.77939444
  6.51430347  6.5947808   6.5947808   6.5947808   6.69232757  6.70261452
  6.70261452  6.70261452  6.77898812  6.77898812  6.77898812  7.62404927
  7.64612613  7.64612613  7.64612613  7.76173783  7.76173783  7.76173783
  8.51540735  8.57938471  8.57938471  8.57938471  8.73608255  8.73608255
  8.73608255  9.42295247  9.69793715  9.69793715  9.69793715 10.01790688
 10.56473012 10.62821763 10.62821763 10.62821763 10.70969514 11.02184359
 11.27609891 11.62936661 12.52912408 13.41298761 14.56514541 15.42614026]


Comparison with Øyvinds code

In [49]:
odqd = ODQD(num_orbitals, x_max, num_points, alpha=1, a=0.01, potential=HOPotential())
system = GeneralOrbitalSystem(num_electrons, odqd)
cisd = CISD(system, verbose=False).compute_ground_state()
print(cisd.energies)
#print(f'Address scheme difference (rounded to 10 decimals):\n{np.abs(np.round(E-cisd.energies, 10))}')
#print(f'Slater condon difference (rounded to 10 decimals):\n{np.abs(np.round(E-cisd.energies, 10))}')

[ 2.69518071  2.73313865  2.73313865  2.73313865  3.70897992  3.73641856
  3.73641856  3.73641856  4.57540917  4.64307892  4.64307892  4.64307892
  4.72949421  4.74023615  4.74023615  4.74023615  5.62018958  5.6594825
  5.6594825   5.6594825   5.73763097  5.77939444  5.77939444  5.77939444
  6.51430347  6.5947808   6.5947808   6.5947808   6.69232757  6.70261452
  6.70261452  6.70261452  6.77898812  6.77898812  6.77898812  7.62404927
  7.64612613  7.64612613  7.64612613  7.76173783  7.76173783  7.76173783
  8.51540735  8.57938471  8.57938471  8.57938471  8.73608255  8.73608255
  8.73608255  9.42295247  9.69793715  9.69793715  9.69793715 10.01790688
 10.56473012 10.62821763 10.62821763 10.62821763 10.70969514 11.02184359
 11.27609891 11.62936661 12.52912408 13.41298761 14.56514541 15.42614026]


Is the address scheme better than using Slater-Condon?

In [42]:
a=num_electrons
b=num_spin_orbitals
c=comb(b,a)
print(f'Number of iterations for Slater-Condon: {c*c:.2e}')
print(f'Number of iterations for address scheme: {c*b*b*a*a:.2e}')

Number of iterations for Slater-Condon: 2.25e+02
Number of iterations for address scheme: 2.16e+03


# Imaginary time propagation

In [43]:
def normalize(psi):
    return psi/np.sqrt(psi@psi)

In [44]:
# Choose a random trial state
rng = np.random.default_rng()
psi_trial =  rng.random(Psi[:,0].size)

# Normalize
psi_trial = normalize(psi_trial)

print(f'Overlap with ground state before propagation: {np.abs(psi_trial@Psi[:,0])}')

# Propagation step size
tau = 1
# Propagation operator
U = expm(-H*tau)

for i in range(10):
    # Propagate
    psi_trial = U@psi_trial
    # Renormalize
    psi_trial = normalize(psi_trial)

# Check that we have converged to the ground state
print(f'Overlap with ground state after propagation: {np.abs(psi_trial@Psi[:,0])}')

Overlap with ground state before propagation: 0.19360029589391772
Overlap with ground state after propagation: 0.999999999559559
